In [0]:
# ============================================================
# FRESH NOTEBOOK (RUN TOP TO BOTTOM)
#
# 4 POPULATIONS (business-verified logic preserved):
# 1) First Time Joiners  : JOIN → annual1 within offer window
# 2) Upgrades            : CHANGE monthly1→monthly1 with upgrade_status within offer window
# 3) Returning           : CHANGE → annual1 (active) within offer window
# 4) Renewals (separate) : Joined LY window AND active annual as-of asof_end
#
# BREAKDOWN REQUIRED:
# - Breakdown the RETURNING population using your bucket logics:
#     Bucket 2: Returning from Gift
#     Bucket 3: Returning from Annual
#     Bucket 4: Returning from Monthly
#   (Bucket 1 stays its own population "Upgrades"; renewals remain separate.)
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings("ignore")

# ----------------------------
# 1) Parameters
# ----------------------------
start_date = "2025-12-11"
end_date   = "2025-12-26"

active_statuses  = ["Active", "Active_Non_Renewing"]
upgrade_statuses = ["Active_Annnual_Upgrade"]
target_plan      = "annual1"

# Renewals definition (separate population)
join_start = "2024-12-11"
join_end   = "2024-12-26"
asof_end   = "2025-12-27"  # as-of cutoff for renewal persistence

# ----------------------------
# 2) Load table
# ----------------------------
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")

# ============================================================
# 3) Build df_events (JOIN + CHANGE) from snapshots
# ============================================================

df_raw_state = (
    T_subscriber
    .withColumn("snapshot_date", F.to_date(F.col("snapshot_date_id").cast("string"), "yyyyMMdd"))
    .select(
        "cde_id",
        "snapshot_date",
        F.lower(F.trim(F.col("plan_id"))).alias("plan_id_norm"),
        F.lower(F.trim(F.col("renewal_status"))).alias("renewal_status_norm"),
        F.col("plan_id").alias("plan_id_raw"),
        F.col("renewal_status").alias("renewal_status_raw"),
        "activated_at_datetime",
        "last_updated"
    )
    .filter(F.col("cde_id").isNotNull())
    .filter(F.col("snapshot_date").isNotNull())
)

w_day = Window.partitionBy("cde_id", "snapshot_date").orderBy(
    F.col("last_updated").desc_nulls_last(),
    F.col("activated_at_datetime").desc_nulls_last()
)

df_state = (
    df_raw_state
    .withColumn("rn_day", F.row_number().over(w_day))
    .filter(F.col("rn_day") == 1)
    .drop("rn_day")
)

w_join = Window.partitionBy("cde_id").orderBy(
    F.col("activated_at_datetime").asc_nulls_last(),
    F.col("snapshot_date").asc(),
    F.col("last_updated").desc_nulls_last()
)

df_join_event = (
    df_state
    .withColumn("rn_join", F.row_number().over(w_join))
    .filter(F.col("rn_join") == 1)
    .select(
        "cde_id",
        F.coalesce(F.col("activated_at_datetime"), F.col("snapshot_date").cast("timestamp")).alias("event_ts"),
        F.lit("JOIN").alias("event_type"),
        F.lit(None).cast("string").alias("from_plan_id"),
        F.col("plan_id_raw").alias("to_plan_id"),
        F.lit(None).cast("string").alias("from_renewal_status"),
        F.col("renewal_status_raw").alias("to_renewal_status"),
        F.col("snapshot_date").alias("observed_snapshot_date"),
        F.col("activated_at_datetime")
    )
)

w = Window.partitionBy("cde_id").orderBy("snapshot_date")

df_change_events = (
    df_state
    .withColumn("prev_snapshot_date", F.lag("snapshot_date").over(w))
    .withColumn("prev_plan_norm",     F.lag("plan_id_norm").over(w))
    .withColumn("prev_status_norm",   F.lag("renewal_status_norm").over(w))
    .withColumn("prev_plan_raw",      F.lag("plan_id_raw").over(w))
    .withColumn("prev_status_raw",    F.lag("renewal_status_raw").over(w))
    .filter(F.col("prev_snapshot_date").isNotNull())
    .filter(
        (F.col("plan_id_norm") != F.col("prev_plan_norm")) |
        (F.col("renewal_status_norm") != F.col("prev_status_norm"))
    )
    .select(
        "cde_id",
        F.col("snapshot_date").cast("timestamp").alias("event_ts"),
        F.lit("CHANGE").alias("event_type"),
        F.col("prev_plan_raw").alias("from_plan_id"),
        F.col("plan_id_raw").alias("to_plan_id"),
        F.col("prev_status_raw").alias("from_renewal_status"),
        F.col("renewal_status_raw").alias("to_renewal_status"),
        F.col("snapshot_date").alias("observed_snapshot_date"),
        F.col("activated_at_datetime")
    )
)

df_events = (
    df_join_event
    .unionByName(df_change_events)
    .dropDuplicates([
        "cde_id", "event_ts", "event_type",
        "from_plan_id", "to_plan_id",
        "from_renewal_status", "to_renewal_status"
    ])
)

# ============================================================
# 4) Population 1: First Time Joiners
# ============================================================
df_first_time_joiners = (
    df_events
    .filter(F.col("event_type") == "JOIN")
    .filter(F.col("event_ts").between(start_date, end_date))
    .filter(F.col("to_plan_id") == target_plan)
    .select("cde_id")
    .distinct()
)

# ============================================================
# 5) Population 2: Upgrades (Bucket 1 logic)
# ============================================================
df_upgrades = (
    df_events
    .filter(F.col("event_type") == "CHANGE")
    .filter(F.col("event_ts").between(start_date, end_date))
    .filter(F.col("from_plan_id") == "monthly1")
    .filter(F.col("to_plan_id") == "monthly1")
    .filter(F.col("to_renewal_status").isin(upgrade_statuses))
    .select("cde_id")
    .distinct()
)



# ============================================================
# 7) Population 4: Renewals (separate)
# ============================================================
df_joined_last_year = (
    df_events
    .filter(F.col("event_type") == "JOIN")
    .filter(F.col("event_ts").between(join_start, join_end))
    .filter(F.col("to_plan_id") == target_plan)
    .select("cde_id")
    .distinct()
)

w_last_asof = Window.partitionBy("cde_id").orderBy(F.col("event_ts").desc(), F.col("event_type").desc())

df_last_event_asof = (
    df_events
    .join(df_joined_last_year, "cde_id", "inner")
    .filter(F.col("event_ts") <= F.lit(asof_end))
    .withColumn("rn", F.row_number().over(w_last_asof))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_renewals = (
    df_last_event_asof
    .filter(F.col("to_plan_id") == target_plan)
    .filter(F.col("to_renewal_status").isin(active_statuses))
    .select("cde_id")
    .distinct()
)

# ============================================================
# 6) Population 3: Returning (entered annual during offer window)
# ============================================================
df_returning_raw = (
    df_events
    .filter(F.col("event_type") == "CHANGE")
    .filter(F.col("event_ts").between(start_date, end_date))
    .filter(F.col("to_plan_id") == target_plan)
    .filter(F.col("to_renewal_status").isin(active_statuses))
    .select("cde_id")
    .distinct()
)

df_exclude_from_returning = (
    df_first_time_joiners.select("cde_id")
    .unionByName(df_upgrades.select("cde_id"))
    .unionByName(df_renewals.select("cde_id"))
    .distinct()
)

df_returning = (
    df_returning_raw
    .join(df_exclude_from_returning, "cde_id", "left_anti")
)
# ----------------------------
# 8) Population counts
# ----------------------------
cnt_joiners  = df_first_time_joiners.count()
cnt_upgrades = df_upgrades.count()
cnt_return   = df_returning.count()
cnt_renewals = df_renewals.count()

print("\n================= POPULATION COUNTS =================")
print(f"Offer window: {start_date} to {end_date} (inclusive)")
print("1) First Time Joiners:", cnt_joiners)
print("2) Upgrades:", cnt_upgrades)
print("3) Returning:", cnt_return)
print(f"4) Renewals (joined {join_start}..{join_end} & active as-of {asof_end}):", cnt_renewals)

# ============================================================
# 9) BREAKDOWN for RETURNING members using your bucket logics (2/3/4)
#    (Do NOT change inherent logic.)
# ============================================================

df_events_returning = df_returning.join(df_events, "cde_id", "inner")

df_events_returning_candidates = (
    df_returning.join(df_events, "cde_id", "inner")
    .filter(F.col("event_type") == "CHANGE")
    .filter(F.col("observed_snapshot_date").between(start_date, end_date))
    .filter(F.col("to_plan_id") == target_plan)
    .filter(F.col("to_renewal_status").isin(active_statuses))
)

df_returning_bucketed_events_candidates = (
    df_events_returning_candidates
    .withColumn(
        "returning_bucket_rank",
        F.when(
            (F.col("from_plan_id") == "gift-plan") &
            (F.col("to_renewal_status").isin(active_statuses)) &
            (F.col("to_plan_id") == target_plan),
            F.lit(2)
        )
        .when(
            (F.col("from_plan_id") == target_plan) &
            (F.col("to_plan_id") == target_plan) &
            (~F.col("from_renewal_status").isin(active_statuses)) &
            (F.col("to_renewal_status").isin(active_statuses)),
            F.lit(3)
        )
        .when(
            (F.col("from_plan_id") == "monthly1") &
            (F.col("to_plan_id") == target_plan) &
            (F.col("to_renewal_status").isin(active_statuses)) &
            (~F.col("to_renewal_status").isin(upgrade_statuses)) &
            (~F.col("from_renewal_status").isin(upgrade_statuses)),
            F.lit(4)
        )
        .otherwise(F.lit(99))
    )
)

df_returning_member_bucket = (
    df_returning_bucketed_events_candidates
    .groupBy("cde_id")
    .agg(F.min("returning_bucket_rank").alias("returning_bucket_rank"))
    .withColumn(
        "returning_bucket",
        F.when(F.col("returning_bucket_rank") == 2, F.lit("Returning from Gift"))
         .when(F.col("returning_bucket_rank") == 3, F.lit("Returning from Annual"))
         .when(F.col("returning_bucket_rank") == 4, F.lit("Returning from Monthly"))
         .otherwise(F.lit("Other annual entry"))
    )
)

df_returning_bucket_counts = (
    df_returning_member_bucket
    .groupBy("returning_bucket")
    .agg(F.countDistinct("cde_id").alias("members"))
    .orderBy("returning_bucket")
)

print("\n================= RETURNING BREAKDOWN (BUCKET 2/3/4) =================")
print("Returning population (denominator):", cnt_return)
print("Counts below should sum to Returning (including 'Other annual entry').")
df_returning_bucket_counts.display()

# Optional: overlaps (informational only)
print("\n================= OPTIONAL OVERLAPS (FYI) =================")
print("Returning ∩ Upgrades:", df_returning.join(df_upgrades, 'cde_id', 'inner').count())
print("Renewals ∩ Returning:", df_renewals.join(df_returning, 'cde_id', 'inner').count())
print("Renewals ∩ Upgrades:", df_renewals.join(df_upgrades, 'cde_id', 'inner').count())


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# A) Build Returning bucket at MEMBER grain (already in your code)
#    df_returning_member_bucket: cde_id, returning_bucket_rank, returning_bucket
# ============================================================

# df_returning_member_bucket is already created in your notebook.
# Ensure it is 1 row per cde_id:
df_returning_member_bucket = df_returning_member_bucket.dropDuplicates(["cde_id"])

# ============================================================
# B) Create a single MECE classification per member
#    Priority (typical, aligns with your intent):
#      1) Upgrades (Bucket 1)  -> separate population
#      2) Renewals            -> separate population
#      3) First Time Joiners
#      4) Returning buckets (Gift/Annual/Monthly/Other)
#      5) Unclassified (should be 0 if your universe is only these pops)
# ============================================================

# Use the UNION of all members you care about as the universe.
# If you want "only members who appear in any of the 4 pops", keep it as below.
df_universe = (
    df_first_time_joiners.select("cde_id")
    .unionByName(df_upgrades.select("cde_id"))
    .unionByName(df_returning.select("cde_id"))
    .unionByName(df_renewals.select("cde_id"))
    .distinct()
)

df_mece = (
    df_universe
    .join(df_upgrades.withColumn("is_upgrade", F.lit(1)), "cde_id", "left")
    .join(df_renewals.withColumn("is_renewal", F.lit(1)), "cde_id", "left")
    .join(df_first_time_joiners.withColumn("is_joiner", F.lit(1)), "cde_id", "left")
    .join(df_returning_member_bucket.select("cde_id", "returning_bucket"), "cde_id", "left")
    .withColumn("is_upgrade", F.coalesce(F.col("is_upgrade"), F.lit(0)))
    .withColumn("is_renewal", F.coalesce(F.col("is_renewal"), F.lit(0)))
    .withColumn("is_joiner",  F.coalesce(F.col("is_joiner"),  F.lit(0)))
    .withColumn(
        "final_category",
        F.when(F.col("is_upgrade") == 1, F.lit("Upgrades"))
         .when(F.col("is_renewal") == 1, F.lit("Renewals"))
         .when(F.col("is_joiner")  == 1, F.lit("First Time Joiners"))
         .when(F.col("returning_bucket").isNotNull(), F.col("returning_bucket"))
         .otherwise(F.lit("Unclassified"))
    )
)

# ============================================================
# C) Validate MECE
# ============================================================

# 1 row per member?
print("Universe members:", df_universe.count())
print("MECE rows:", df_mece.count())

# Counts by final category
df_mece_counts = (
    df_mece.groupBy("final_category")
    .agg(F.countDistinct("cde_id").alias("members"))
    .orderBy("final_category")
)

df_mece_counts.display()

# Optional: check overlap flags to see how many members were multi-eligible
df_overlap_diagnostics = (
    df_mece
    .select(
        "cde_id",
        "final_category",
        "is_upgrade",
        "is_renewal",
        "is_joiner",
        F.when(F.col("returning_bucket").isNotNull(), 1).otherwise(0).alias("is_returning")
    )
    .withColumn("eligibility_count", F.col("is_upgrade") + F.col("is_renewal") + F.col("is_joiner") + F.col("is_returning"))
)

(df_overlap_diagnostics
 .groupBy("eligibility_count")
 .agg(F.countDistinct("cde_id").alias("members"))
 .orderBy("eligibility_count")
).display()


In [0]:
df_returning_bucketed_events_candidates \
    .filter(F.col("returning_bucket_rank") == 2) \
    .groupBy(
        "from_plan_id",
        "to_plan_id",
        "from_renewal_status",
        "to_renewal_status"
    ) \
    .agg(F.countDistinct("cde_id").alias("members")) \
    .orderBy(F.desc("members")) \
    .display()


In [0]:
T_demo = spark.table('cpx_dataanalytics_gold.customer360.cpx_d_customer')
T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
snapshot_date = '20251227'

# Demographics snapshot
df_demographics_main = (
    T_subscriber.join(T_demo, "cde_id", "left")
    .filter(F.col("snapshot_date_id") == snapshot_date)
    .select(
        "cde_id",
        "plan_id",
        "renewal_status",
        "date_of_birth_masked",
        "gender_text",
        "province",
        T_subscriber.cc_user_profile_id
    )
    .dropDuplicates(["cde_id"])
)

# Transactions (visits) main
w_cust = Window.partitionBy("cde_id")

df_trans_main = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_films.select("film_id", "title_name"), T_trans.film_id == T_films.film_id, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre") &
        (T_trans.CDE_ID.isNotNull())
    )
    .withColumnRenamed("CDE_ID", "cde_id")
    .groupBy("cde_id", "datefull", T_trans.LocationId, "title_name")
    .agg(F.sum(F.col("Quantity")).alias("Net_Quantity"))
    .filter(F.col("Net_Quantity") > 0)
    .withColumn("Total_Visits", F.count(F.lit(1)).over(w_cust))
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# ONE BLOCK: P12M refund-correct visits + per-member totals + bins + summary + distribution
# Visit definition: 1 member + 1 date + 1 location
# Refund correction: sum(Quantity) at visit grain; keep only Net_Quantity > 0
# ============================================================

# ----------------------------
# Parameters
# ----------------------------
asof_date = end_date  # e.g., "2025-12-26"
asof      = F.to_date(F.lit(asof_date))
p12_start = F.trunc(F.add_months(asof, -11), "MM")   # last 12 calendar months inclusive
p12_end   = asof

w_cust = Window.partitionBy("cde_id")
w_all  = Window.partitionBy()

# ----------------------------
# 1) P12M refund-correct visit table (visit grain)
# ----------------------------
df_trans_p12_visits = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(
        T_date.select(F.col("dateid").alias("VisitDateId_join"), "datefull"),
        T_trans.VisitDateId == F.col("VisitDateId_join"),
        "left"
    )
    .filter(
        (F.col("LC_Location_Type_Description") == "Cineplex Theatre") &
        (F.col("CDE_ID").isNotNull())
    )
    .select(
        F.col("CDE_ID").alias("cde_id"),
        F.to_date("datefull").alias("datefull"),
        T_trans.LocationId,
        F.col("Quantity").cast("double").alias("Quantity")
    )
    .filter(F.col("datefull").between(p12_start, p12_end))     # P12M scope BEFORE agg
    .groupBy("cde_id", "datefull", T_trans.LocationId)
    .agg(F.sum("Quantity").alias("Net_Quantity"))
    .filter(F.col("Net_Quantity") > 0)                         # refund-correct visit
)

# ----------------------------
# 2) Per-member P12M visit totals
# ----------------------------
df_member_visits_p12m = (
    df_trans_p12_visits
    .groupBy("cde_id")
    .agg(F.count(F.lit(1)).alias("Total_Visits_P12M"))          # each row = 1 visit
)

# ----------------------------
# 3) Full analysis population (denominator) + explicit zeros
#     Assumes these 4 DF exist: df_first_time_joiners, df_upgrades, df_returning, df_renewals
# ----------------------------

returning_buckets_in_scope = [
    "Returning from Gift",
    "Returning from Annual",
    "Returning from Monthly",
    # "Other annual entry",   # include only if you want it in the population
]

df_returning_in_scope = (
    df_returning_member_bucket
    .filter(F.col("returning_bucket").isin(returning_buckets_in_scope))
    .select("cde_id")
    .distinct()
)

df_analysis_population = (
    df_first_time_joiners.select("cde_id")
    .unionByName(df_upgrades.select("cde_id"))
    .unionByName(df_returning_in_scope.select("cde_id"))
    .unionByName(df_renewals.select("cde_id"))
    .distinct()
)

df_visits_full_p12m = (
    df_analysis_population
    .select("cde_id").distinct()
    .join(df_member_visits_p12m, "cde_id", "left")
    .withColumn("Total_Visits_P12M", F.coalesce(F.col("Total_Visits_P12M"), F.lit(0)))
)

# ----------------------------
# 4) Visit buckets (you can rename labels if needed)
# ----------------------------
df_visits_binned_p12m = (
    df_visits_full_p12m
    .withColumn(
        "visit_bin",
        F.when(F.col("Total_Visits_P12M") == 0, "0 Visits")
         .when(F.col("Total_Visits_P12M") == 1, "1 Visit")
         .when((F.col("Total_Visits_P12M") >= 2) & (F.col("Total_Visits_P12M") <= 3), "2-3 Visits")
         .when((F.col("Total_Visits_P12M") >= 4) & (F.col("Total_Visits_P12M") <= 6), "4-6 Visits")
         .when((F.col("Total_Visits_P12M") >= 7) & (F.col("Total_Visits_P12M") <= 12), "7-12 Visits")
         .otherwise("13+ Visits")
    )
    .withColumn(
        "bin_order",
        F.when(F.col("visit_bin") == "0 Visits", 0)
         .when(F.col("visit_bin") == "1 Visit", 1)
         .when(F.col("visit_bin") == "2-3 Visits", 2)
         .when(F.col("visit_bin") == "4-6 Visits", 3)
         .when(F.col("visit_bin") == "7-12 Visits", 4)
         .otherwise(5)
    )
)

# ----------------------------
# 5) Distribution (members + proportions) + summary metrics
# ----------------------------
df_p12m_distribution = (
    df_visits_binned_p12m
    .groupBy("visit_bin", "bin_order")
    .agg(F.countDistinct("cde_id").alias("members"))
    .withColumn("population", F.sum("members").over(w_all))
    .withColumn("proportion", F.round(F.col("members") / F.col("population"), 4))
    .orderBy("bin_order")
    .drop("bin_order", "population")
)

df_p12m_summary = (
    df_visits_full_p12m
    .agg(
        F.countDistinct("cde_id").alias("population"),
        F.sum("Total_Visits_P12M").alias("total_visits_p12m"),
        F.avg("Total_Visits_P12M").alias("avg_visits_per_customer_including_zeros")
    )
    .withColumn("asof_date", F.lit(asof_date))
    .withColumn("window_start", F.date_format(p12_start, "yyyy-MM-dd"))
    .withColumn("window_end", F.date_format(p12_end, "yyyy-MM-dd"))
    .select("asof_date", "window_start", "window_end", "population", "total_visits_p12m", "avg_visits_per_customer_including_zeros")
)

# ----------------------------
# Output
# ----------------------------
df_p12m_summary.display()
df_p12m_distribution.display()

# Optional: member-level table if you need it later
# df_visits_binned_p12m.display()


In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# TOP 10 FILMS WATCHED BY THE ANALYSIS POPULATION (OFFER PERIOD)
# ---------------------------------------------------------
df_pop_trans_offer = (
    T_trans
    .join(df_analysis_population.select(F.col("cde_id").alias("pop_cde_id")),
          T_trans.CDE_ID == F.col("pop_cde_id"),
          "inner")
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(
        T_date.select(F.col("dateid").alias("VisitDateId_join"), "datefull"),
        T_trans.VisitDateId == F.col("VisitDateId_join"),
        "left"
    )
    # Replace T_trans.FilmId with your true film id column in T_trans
    .join(T_films.select("film_id", "title_name"),
          T_trans.film_id == T_films.film_id,
          "left")
    .filter(
        (F.col("LC_Location_Type_Description") == "Cineplex Theatre") &
        (F.to_date("datefull").between(F.lit(start_date), F.lit(end_date)))
    )
    .select(
        F.col("pop_cde_id").alias("cde_id"),
        "title_name",
        F.to_date("datefull").alias("datefull"),
        F.col("Quantity").cast("double").alias("Quantity")
    )
)

# Refund-correct: aggregate to visit grain first
df_pop_visits_offer = (
    df_pop_trans_offer
    .groupBy("cde_id", "datefull", "title_name")
    .agg(F.sum("Quantity").alias("Net_Quantity"))
    .filter(F.col("Net_Quantity") > 0)
    .filter(F.col("title_name").isNotNull())
)

# Aggregate to film-level counts
df_top_films_offer = (
    df_pop_visits_offer
    .groupBy("title_name")
    .agg(
        F.countDistinct("cde_id").alias("unique_members"),
        F.count("*").alias("visit_count")   # counts (member, day, title) occurrences
    )
    .orderBy(F.col("visit_count").desc())
    .limit(10)
)

df_top_films_offer.display()


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from IPython.display import Markdown, display

w = Window.partitionBy()

# ============================================================
# Gender breakdown for ANALYSIS POPULATION
# Assumes:
#   df_analysis_population: has cde_id
#   df_demographics_main: has cc_user_profile_id, gender_text
# ============================================================

def show_gender_distribution(df_in, title, total_label, id_col="cde_id", gender_col="gender_text"):
    display(Markdown(f"**{title}**"))

    # Distinct people denominator (safer than row count)
    df_base = (
        df_in
        .select(F.col(id_col).alias("cde_id"), F.col(gender_col).alias("gender_text"))
        .dropDuplicates(["cde_id"])
    )

    total = df_base.select("cde_id").distinct().count()
    print(total_label, total)

    (
        df_base
        .withColumn("gender_clean", F.initcap(F.lower(F.trim(F.col("gender_text")))))
        .filter(F.col("gender_clean").isin("Male", "Female"))
        .groupBy("gender_clean")
        .agg(F.countDistinct("cde_id").alias("members"))
        .withColumn("proportion", F.round(F.col("members") / F.sum("members").over(w), 4))
        .orderBy("gender_clean")
        .display()
    )

# -------------------------
# Analysis population + demographics
# -------------------------
df_analysis_population_demo = (
    df_analysis_population
    .select("cde_id")
    .distinct()
    .join(
        df_demographics_main.select(
            F.col("cde_id"),
            "gender_text"
        ),
        "cde_id",
        "left"
    )
)

show_gender_distribution(
    df_analysis_population_demo,
    "% Gender Distribution (Analysis Population)",
    "Analysis Population:"
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from IPython.display import Markdown, display

w_all = Window.partitionBy()

def show_age_distribution(df_in, title, total_label, id_col="cde_id", dob_col="date_of_birth_masked"):
    display(Markdown(f"**{title}**"))

    # Distinct people denominator (safer than row count)
    df_base = (
        df_in
        .select(F.col(id_col).alias("cde_id"), F.col(dob_col).alias(dob_col))
        .dropDuplicates(["cde_id"])
    )

    total_people = df_base.select("cde_id").distinct().count()
    print(total_label, total_people)

    age_years = F.year(F.current_date()) - F.col(dob_col).cast("int")

    age_bin_expr = (
        F.when(age_years >= 65, "65+")
         .when(age_years >= 50, "50-64")
         .when(age_years >= 35, "35-49")
         .when(age_years >= 25, "25-34")
         .when(age_years >= 19, "19-24")
         .when(age_years >= 14, "14-18")
         .otherwise("NULL")
    )

    df_bins = (
        df_base
        .withColumn("age_years", age_years)
        .withColumn("age_bin", age_bin_expr)
        .groupBy("age_bin")
        .agg(F.countDistinct("cde_id").alias("count"))
        # denominator excludes NULL
        .withColumn(
            "valid_total",
            F.sum(F.when(F.col("age_bin") != "NULL", F.col("count")).otherwise(0)).over(w_all)
        )
        .withColumn(
            "proportion_pct",
            F.when(
                F.col("age_bin") != "NULL",
                F.round(F.col("count") / F.col("valid_total"), 4)
            )
        )
        .orderBy(
            F.when(F.col("age_bin") == "14-18", 1)
             .when(F.col("age_bin") == "19-24", 2)
             .when(F.col("age_bin") == "25-34", 3)
             .when(F.col("age_bin") == "35-49", 4)
             .when(F.col("age_bin") == "50-64", 5)
             .when(F.col("age_bin") == "65+", 6)
             .otherwise(99)
        )
    )

    df_bins.select("age_bin", "count", "proportion_pct").display()


df_analysis_population_demo = (
    df_analysis_population
    .select("cde_id")
    .distinct()
    .join(
        df_demographics_main.select(
            F.col("cde_id"),
            "date_of_birth_masked"
        ),
        "cde_id",
        "left"
    )
)

show_age_distribution(
    df_analysis_population_demo,
    "% Age Distribution (Analysis Population)",
    "Analysis Population:",
    id_col="cde_id",
    dob_col="date_of_birth_masked"
)


In [0]:
get_ids = df_analysis_population\
    .select("cde_id")\
    .distinct()

get_ids.display()

In [0]:
get_ids = df_analysis_population\
    .select("cde_id")\
    .distinct()\
        .join(spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address"), "cde_id", "inner")


In [0]:
get_ids.display()

In [0]:
df_analysis_population_demo = (
    df_analysis_population
    .select("cde_id")
    .distinct()
    .join(
        df_demographics_main.select(
            F.col("cde_id"),
            "province"
        ),
        "cde_id",
        "left"
    )
)


from pyspark.sql import functions as F
from pyspark.sql.window import Window
from IPython.display import Markdown, display

w = Window.partitionBy()  # global window

def show_province_distribution(df_in, title, total_label, province_col="province"):
    display(Markdown(f"**{title}**"))

    # Provinces to collapse into ATL
    atl_provinces = ["NB", "NL", "NS", "PE", "PEI"]

    # Provinces / states / invalid codes to exclude (set to NULL)
    invalid_provinces = [
        "NT", "YT", "NU", "WRONG",
        "OH", "FL", "WA", "MA", "GA", "AZ", "NC", "KS",
        "CA", "MD", "NJ", "AL", "MI", "VA", "OR", "TX",
        "NY", "IL", "SC", "MN", "VT", "UT", "NV", "MT",
        "PA", "AR", "KY", "CT", "TN", "IA", "DC", "OK", "NH", "CO",
        "WV", "IN", "DE", "LA", "ME", "ND"
    ]

    df_norm = (
        df_in
        .withColumn(
            "Province",
            F.when(F.upper(F.trim(F.col(province_col))).isin(atl_provinces), "ATL")
             .when(F.upper(F.trim(F.col(province_col))).isin(invalid_provinces), None)
             .otherwise(F.upper(F.trim(F.col(province_col))))
        )
    )

    # Exclude NULL / blank provinces
    df_non_null = df_norm.filter(F.col("Province").isNotNull() & (F.col("Province") != ""))

    # Build counts + proportions
    df_counts = (
        df_non_null
        .groupBy("Province")
        .agg(F.count("*").alias("count"))
        .withColumn("proportion", F.round(F.col("count") / F.sum("count").over(w), 4))
    )

    # Total (aligned to denominator used in proportions)
    total = df_non_null.count()
    print(total_label, total)

    # Custom display order
    order_list = ["ON", "QC", "BC", "AB", "MB", "SK", "ATL"]
    order_arr  = F.array(*[F.lit(x) for x in order_list])

    df_out = (
        df_counts
        .withColumn("sort_key", F.array_position(order_arr, F.col("Province")))
        .withColumn("sort_key", F.when(F.col("sort_key") == 0, 999).otherwise(F.col("sort_key")))
        .orderBy("sort_key")
        .drop("sort_key")
    )

    df_out.display()


# -------------------------
# (A) Province Distribution (Month Signups)
# -------------------------
show_province_distribution(
    df_analysis_population_demo,
    "% Province Distribution (Month Signups)",
    "Signups:",
    province_col="province"
)



In [0]:
import pandas as pd

In [0]:
pip install openpyxl

In [0]:
import pandas as pd

pdf = pd.read_excel(
    "/Workspace/Users/ranjan.mishra@cineplex.com/Codebase/CineClub/Offers Post Mortems/CineClub Winter Holiday Offer 2025/Gift_Customer_ids.xlsx",
    sheet_name="Sheet1"
)

cc_user = spark.createDataFrame(pdf)
cc_user_2 = cc_user.withColumnRenamed(
    "invoice.cc_user_profile_id",
    "cc_user_profile_id"
).distinct()

display(cc_user.count())
display(cc_user_2.count())

In [0]:
from pyspark.sql import functions as F

cc_user_3 = (
    cc_user_2
    .join(
        T_subscriber,
        "cc_user_profile_id",
        "left"
    )
    .select("cde_id")
    .distinct()
)

display(cc_user_3.count())

In [0]:
cc_user_3 = cc_user_3\
        .join(spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address"), "cde_id", "inner")\
        .dropDuplicates(["cde_id"])


In [0]:
cc_user_3.display()